<a href="https://colab.research.google.com/github/garimamishra1017/Crop-Yield-And-disease-prediction/blob/main/Crop_Yield_And_disease_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Path to dataset files: /kaggle/input/plantvillage-dataset


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving crop_yield.csv to crop_yield (1).csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Sentinel-2 Imagery_ NDVI Colormap.jpg to Sentinel-2 Imagery_ NDVI Colormap (1).jpg


In [ ]:
!pip install torch torchvision scikit-learn pandas rasterio streamlit joblib pyngrok

Synthetic Dataset Generator (for quick testing)

In [ ]:
import pandas as pd
import numpy as np

# Generate synthetic weather + soil + yield dataset
np.random.seed(42)
rows = 200
df = pd.DataFrame({
    "rainfall": np.random.uniform(50, 200, rows),
    "temperature": np.random.uniform(15, 35, rows),
    "humidity": np.random.uniform(40, 90, rows),
    "soil_moisture": np.random.uniform(10, 40, rows),
    "yield": np.random.uniform(1.5, 4.5, rows)  # tons/ha
})
df.to_csv("weather_soil.csv", index=False)
print("Synthetic dataset saved as weather_soil.csv")
df.head()


Synthetic dataset saved as weather_soil.csv


,rainfall,temperature,humidity,soil_moisture,yield
0,106.181018,27.840633,45.156193,15.068052,3.621716
1,192.607146,16.682799,85.127645,18.357710,1.957617
2,159.799091,18.232574,65.262619,15.310315,3.228865
3,139.798773,32.971084,81.322873,12.661076,3.320145
4,73.402796,27.128581,56.002480,13.619076,2.772392


NDVI Preprocessing (Satellite Imagery)

In [ ]:
import rasterio
import numpy as np

def compute_ndvi(red_band, nir_band):
    return (nir_band - red_band) / (nir_band + red_band + 1e-10)

def load_satellite_image(path):
    with rasterio.open(path) as src:
        red = src.read(3).astype(float)   # Sentinel-2 band 3
        nir = src.read(8).astype(float)   # Sentinel-2 band 8
    return compute_ndvi(red, nir)


CNN for Disease Detection (demo training)

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models

def train_disease_cnn(data_dir, epochs=1):  # keep small for demo
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])
    dataset = datasets.ImageFolder(data_dir, transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        for images, labels in loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    torch.save(model.state_dict(), "cnn_disease.pth")
    return model

# Example run (requires PlantVillage dataset uploaded to /content/leaf_disease)
# train_disease_cnn("/content/leaf_disease")


Yield Forecasting Model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import joblib

def train_yield_model(csv_path="weather_soil.csv"):
    df = pd.read_csv(csv_path)
    X = df[['rainfall','temperature','humidity','soil_moisture']]
    y = df['yield']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    model = GradientBoostingRegressor()
    model.fit(X_train, y_train)

    joblib.dump(model, "yield_regressor.pkl")
    print("Yield model trained and saved.")
    return model

# Train on synthetic dataset
train_yield_model("weather_soil.csv")


Yield model trained and saved.


GradientBoostingRegressor()

In [ ]:
from pyngrok import ngrok

# Paste your token here
!ngrok config add-authtoken 3Fls9OUkSR3fidJJl863szowCfg_6fyharras7hCfBn63qvkr


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!streamlit run dashboard.py &>/dev/null&
public_url = ngrok.connect(8501)
print("Dashboard URL:", public_url)


Dashboard URL: NgrokTunnel: "https://ruckus-protegee-escalator.ngrok-free.dev" -> "http://localhost:8501"


Farmer Dashboard (Streamlit + ngrok)

In [ ]:

with open("dashboard.py", "w") as f:
    f.write("""
import streamlit as st
import joblib
import torch
from PIL import Image
import torchvision.transforms as transforms
from torchvision import models

yield_model = joblib.load("yield_regressor.pkl")
disease_model = models.resnet18()
disease_model.fc = torch.nn.Linear(disease_model.fc.in_features, 2)
disease_model.eval()

st.title("🌾 Crop Yield & Disease Prediction Dashboard")

st.header("Leaf Disease Detection")
uploaded_file = st.file_uploader("Upload leaf image", type=["jpg","png"])
if uploaded_file:
    # Force conversion to RGB (fixes 4-channel error)
    image = Image.open(uploaded_file).convert("RGB")
    transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        prediction = disease_model(input_tensor)
    st.write("Disease Prediction:", torch.argmax(prediction).item())

st.header("Yield Forecasting")
rainfall = st.number_input("Rainfall (mm)")
temperature = st.number_input("Temperature (°C)")
humidity = st.number_input("Humidity (%)")
soil_moisture = st.number_input("Soil Moisture (%)")

if st.button("Predict Yield"):
    features = [[rainfall, temperature, humidity, soil_moisture]]
    forecast = yield_model.predict(features)[0]
    st.success(f"Predicted Yield: {forecast:.2f} tons/ha")
""")


In [ ]:
from pyngrok import ngrok

!streamlit run dashboard.py &>/dev/null&
public_url = ngrok.connect(8501)
print("Dashboard URL:", public_url)


Dashboard URL: NgrokTunnel: "https://ruckus-protegee-escalator.ngrok-free.dev" -> "http://localhost:8501"
